In [1]:
import pandas as pd
import json
import ast

### Load data

In [ ]:
file_path = '/data/TriBERT-Out_of_Domain_Evaluation/data.xlsx'
df = pd.read_excel(file_path)
df.head()

### Convert TriBERT to our format

In [3]:
def calculate_intervals(data, split_type):
    char_position = 0
    word_position = 0

    char_intervals = []
    word_intervals = []
    sep_indices = []

    for text, label in data:
        word_count = len(text.split())
        char_count = len(text)
        
        if label == 'machine':
            char_start = char_position
            char_end = char_position + char_count
            
            word_start = word_position
            word_end = word_position + word_count - 1

            if char_intervals and char_intervals[-1][1] + 1 == char_start:
                char_intervals[-1] = [char_intervals[-1][0], char_end]
                word_intervals[-1] = [word_intervals[-1][0], word_end]
            else:
                char_intervals.append([char_start, char_end])
                word_intervals.append([word_start, word_end])

        char_position += char_count + 1
        word_position += word_count
    
    if split_type == "H_M":
        sep_indices.append(char_intervals[0][0])
    elif split_type == "M_H":
        sep_indices.append(char_intervals[0][1])
    elif split_type == "H_M_H":
        sep_indices.append(char_intervals[0][0])
        sep_indices.append(char_intervals[0][1])
    elif split_type == "M_H_M":
        sep_indices.append(char_intervals[0][1])
        sep_indices.append(char_intervals[1][0])
    elif split_type == "H_M_H_M":
        sep_indices.append(char_intervals[0][0])
        sep_indices.append(char_intervals[0][1])
        sep_indices.append(char_intervals[1][0])
    elif split_type == "M_H_M_H":
        sep_indices.append(char_intervals[0][1])
        sep_indices.append(char_intervals[1][0])
        sep_indices.append(char_intervals[1][1])
            
    return char_intervals, word_intervals, sep_indices


In [ ]:
### Check interval converting
text_num = 11815
text = eval(df.iloc[text_num]['sent_and_label'])
split_type = df.iloc[text_num]['author_seq']
print(text)
print(split_type)
calculate_intervals(text, split_type)

In [5]:
def convert_func(dataframe):
    
    out_jsonl = []
    for i in range(len(dataframe)):
        
        data_human = {
            "label": "human",
            "model": "human",
            "text":  dataframe.iloc[i]['human_part'],
            "source": "TriBERT",
            "source_dataset": "TriBERT",
            "essayset": int(dataframe.iloc[i]['essayset']),
            "data_type": "article",
            "original_split": dataframe.iloc[i]['train_ix'],
            "topic_id": "empty",
            "prompt": "empty",
            "ended": True,
        }
        
        data_ai = {
            "label": "ai",
            "model": "gpt",
            "text":  dataframe.iloc[i]['machine_part'],
            "source": "TriBERT",
            "source_dataset": "TriBERT",
            "essayset": int(dataframe.iloc[i]['essayset']),
            "data_type": "article",
            "original_split": dataframe.iloc[i]['train_ix'],
            "topic_id": "empty",
            "prompt": "empty",
            "prompt_type": "machine_specified",
            "ended": True,
        }
        
        text = eval(df.iloc[i]['sent_and_label'])
        split_type = df.iloc[i]['author_seq']
        ai_char_intervals, ai_words_intervals, sep_indices = calculate_intervals(text, split_type)
        
        data_mixed = {
            "label": "mixed",
            "model": "gpt",
            "text":  dataframe.iloc[i]['hybrid_text'],
            "source": "TriBERT",
            "source_dataset": "TriBERT",
            "essayset": int(dataframe.iloc[i]['essayset']),
            "data_type": "article",
            "original_split": dataframe.iloc[i]['train_ix'],
            "topic_id": "empty",
            "prompt": "empty",
            "prompt_type": "machine_specified",
            "ended": True,
            "ai_char_intervals": ai_char_intervals,
            "boundary_ix": ast.literal_eval(dataframe.iloc[i]['boundary_ix']),
            "sep_indices": sep_indices,
            "split": eval(dataframe.iloc[i]['sent_and_label']),
        }
        
        out_jsonl.append(data_human)
        # out_jsonl.append(data_ai)
        out_jsonl.append(data_mixed)
    
    return out_jsonl
    

### In-DOMAIN

In [14]:
jsonl_out = convert_func(df)

train_jsonl = []
valid_jsonl = []
test_jsonl = []

for json_str in jsonl_out:
    if json_str['original_split'] == 'train':
        train_jsonl.append(json_str)
    elif json_str['original_split'] == 'valid':
        valid_jsonl.append(json_str)
    elif json_str['original_split'] == 'test':
        test_jsonl.append(json_str)

### Out-Of-Domain

In [ ]:
essayset = 8 # set from 1 to 8
jsonl_out = convert_func(df)

train_jsonl = []
valid_jsonl = []
test_jsonl = []


for json_str in jsonl_out:
    if json_str['original_split'] == 'train' and json_str['essayset'] != essayset:
        train_jsonl.append(json_str)
    elif json_str['original_split'] == 'valid' and json_str['essayset'] != essayset:
        valid_jsonl.append(json_str)
    elif json_str['original_split'] == 'test' and json_str['essayset'] == essayset:
        test_jsonl.append(json_str)

### Save data

In [11]:
def save_to_jsonl(data, filename):
    try:
        with open(filename, 'w', encoding='utf-8') as file:
            for item in data:
                file.write(json.dumps(item) + '\n')
        print(f"Data save in {filename}")
    except Exception as e:
        print(f"Mistake while saving in {filename}: {e}")

In [ ]:
path = "/data/TriBERT/OOD/8/"
save_to_jsonl(train_jsonl, path + "train.jsonl")
save_to_jsonl(valid_jsonl, path + "valid.jsonl")
save_to_jsonl(test_jsonl, path + "test.jsonl")